In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

# ============ Config ============
experiment_name = "strong_aug"
log_dir = f"runs/{experiment_name}"
save_path = f"runs/{experiment_name}/best.pth"
num_epochs = 50
batch_size = 64
learning_rate = 0.001
# ================================

# 1. Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. TensorBoard SummaryWriter
writer = SummaryWriter(log_dir)

# 3. Transform
transform = transforms.Compose([
    transforms.RandomResizedCrop(32, scale=(0.8, 1.2)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(0.5, 0.5, 0.5, 0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomAffine(degrees=20, translate=(0.1, 0.1), scale=(0.8, 1.2)),
    transforms.RandomPerspective(distortion_scale=0.5, p=0.5),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 4. Dataset & DataLoader
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

# 5. Model (fix warning: use weights=None)
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 10)
model.to(device)

# 6. Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 7. Evaluation function
def evaluate():
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

# 8. Training function
def train_one_epoch(epoch):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

    avg_loss = running_loss / len(train_loader)
    train_acc = correct / total
    val_acc = evaluate()

    # Log to TensorBoard
    writer.add_scalar("Loss/train", avg_loss, epoch)
    writer.add_scalar("Accuracy/train", train_acc, epoch)
    writer.add_scalar("Accuracy/val", val_acc, epoch)

    print(f"Epoch [{epoch+1}] Train Loss: {avg_loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")
    return val_acc

# 9. Training loop with best model saving
best_val_acc = 0.0
for epoch in range(num_epochs):
    val_acc = train_one_epoch(epoch)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_path)
        print(f"✅ New best model saved with val acc: {best_val_acc:.4f}")

writer.close()

Epoch [1] Train Loss: 1.8549, Train Acc: 0.3208, Val Acc: 0.3786
✅ New best model saved with val acc: 0.3786
Epoch [2] Train Loss: 1.6195, Train Acc: 0.4140, Val Acc: 0.4362
✅ New best model saved with val acc: 0.4362
Epoch [3] Train Loss: 1.5006, Train Acc: 0.4591, Val Acc: 0.4503
✅ New best model saved with val acc: 0.4503
Epoch [4] Train Loss: 1.4129, Train Acc: 0.4985, Val Acc: 0.5071
✅ New best model saved with val acc: 0.5071
Epoch [5] Train Loss: 1.3377, Train Acc: 0.5250, Val Acc: 0.5386
✅ New best model saved with val acc: 0.5386
Epoch [6] Train Loss: 1.2834, Train Acc: 0.5432, Val Acc: 0.5465
✅ New best model saved with val acc: 0.5465
Epoch [7] Train Loss: 1.2305, Train Acc: 0.5633, Val Acc: 0.5487
✅ New best model saved with val acc: 0.5487
Epoch [8] Train Loss: 1.1933, Train Acc: 0.5766, Val Acc: 0.5657
✅ New best model saved with val acc: 0.5657
Epoch [9] Train Loss: 1.1567, Train Acc: 0.5928, Val Acc: 0.5945
✅ New best model saved with val acc: 0.5945
Epoch [10] Train Lo